In [198]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [199]:
RANDOM_STATE = 42
TARGET = "employed_status"
ID_COL = "anonymised_id"
N_SPLITS = 5

train = pd.read_csv("data/train.csv")
train = train.dropna(subset=["employed_status"])
test = pd.read_csv("data/test.csv")

Feature Engineering: Tenure, gated by prior employment and age x employed_lag and work_readiness_score x is_first_round

In [200]:
def engineer_features(df: pd.DataFrame, quintile_median=None) -> pd.DataFrame:
    df = df.copy()

    df["has_history"] = df["lag_round"].notna().astype(int)
    df["employed_lag_num"] = df["employed_lag"]  # 0/1/NaN
    df["employed_lag_x_recency"] = df["employed_lag_num"].fillna(0) * df["has_history"]

    # --- Tenure, gated by prior employment ---------------------------------
    # only treat tenure as a risk signal for rows that WERE employed last
    # round -- zero it out otherwise, so the coefficient isn't diluted by
    # unrelated non-employed zeros.
    df["tenure_lag_missing"] = df["tenure_lag"].isna().astype(int)
    df["tenure_lag"] = df["tenure_lag"].fillna(0)
    df["log_tenure_lag"] = np.log1p(df["tenure_lag"].clip(lower=0))
    df["log_tenure_lag_if_employed"] = df["log_tenure_lag"] * df["employed_lag_num"].fillna(0)

    df["is_first_round"] = (df["total_historical_rounds"] <= 1).astype(int)

     # --- FIX: Center age before squaring ---------------------------------
    df["age"] = df["age"].fillna(df["age"].median())
    age_mean = df["age"].mean()  # Calculate mean after imputation
    df["age_centered"] = df["age"] - age_mean  # ← NEW
    df["age_sq"] = df["age_centered"] ** 2  # ← CHANGED: square centered age

    # --- age x employed_lag --------------------------------------------
    # Age likely means something different depending on prior state: among
    # the already-employed it's closer to a tenure/seniority proxy; among
    # the not-employed it's closer to a "how long searching" proxy.
    df["age_x_employed_lag"] = df["age"] * df["employed_lag_num"].fillna(0)

    # --- work_readiness_score x is_first_round --------------------------
    # Purpose-built forward-looking score should matter most when it's the
    # ONLY forward signal available (no employed_lag / status history).
    df["work_readiness_x_first_round"] = df["work_readiness_score"].fillna(
        df["work_readiness_score"].median()
    ) * df["is_first_round"]
    
   

    return df

In [ ]:
def add_seasonality_features(df: pd.DataFrame, date_col: str = "survey_date") -> pd.DataFrame:
    """Add cyclical seasonality features: sin/cos of month."""
    df = df.copy()
    
    if date_col not in df.columns:
        df["month_sin"] = 0
        df["month_cos"] = 0
        df["month_sin_x_employed_lag"] = 0
        df["month_cos_x_employed_lag"] = 0
        return df
    
    # Extract month
    month = pd.to_datetime(df[date_col]).dt.month
    
    # Cyclical encoding
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)
    
    # Interaction with employed_lag - season affects transitions differently
    df["month_sin_x_employed_lag"] = df["month_sin"] * df["employed_lag_num"].fillna(0)
    df["month_cos_x_employed_lag"] = df["month_cos"] * df["employed_lag_num"].fillna(0)

    
    return df

In [202]:
def make_interaction_categorical(df, col_a, col_b, new_col, min_count=None,
                                  train_ref=None, other_label="Other"):
    """Combine two categorical columns into one 'A||B' categorical.
    NaNs are stringified so 'Missing' combinations are preserved as their
    own category rather than dropped."""
    a = df[col_a].astype(str).fillna("Missing")
    b = df[col_b].astype(str).fillna("Missing")
    df[new_col] = a + "||" + b

    if min_count is not None:
        ref = train_ref if train_ref is not None else df
        counts = ref[new_col].value_counts()
        keep = set(counts[counts >= min_count].index)
        df[new_col] = df[new_col].where(df[new_col].isin(keep), other_label)
    return df

In [203]:
def add_frequency_encoding(train_df, test_df, col, new_col=None):
    new_col = new_col or f"{col}_freq"
    freq_map = train_df[col].value_counts(normalize=True)
    train_df[new_col] = train_df[col].map(freq_map).fillna(0)
    test_df[new_col] = test_df[col].map(freq_map).fillna(0)
    return train_df, test_df


def collapse_rare_categories(train_df, test_df, col, min_count=30, other_label="Other"):
    counts = train_df[col].value_counts()
    keep = set(counts[counts >= min_count].index)

    def _collapse(series):
        return series.where(series.isin(keep) | series.isna(), other_label)

    train_df[col] = _collapse(train_df[col])
    test_df[col] = _collapse(test_df[col])
    return train_df, test_df

def extract_month_from_date(df: pd.DataFrame, date_col: str = "survey_date") -> pd.Series:
    """Extract month from survey_date column."""
    if date_col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_datetime(df[date_col]).dt.month

In [204]:
def kfold_target_encode(train_df, col, target_col, groups, n_splits=N_SPLITS,
                         smoothing=1, random_state=RANDOM_STATE):
    """Leakage-safe target encoding: out-of-fold for train rows, full-train
    mapping for anything else. Smoothing shrinks small-count categories
    toward the global mean so low-n municipalities don't get noisy extreme
    rates."""
    global_mean = train_df[target_col].mean()
    oof = pd.Series(index=train_df.index, dtype=float)

    gkf = GroupKFold(n_splits=n_splits)
    for tr_idx, val_idx in gkf.split(train_df, train_df[target_col], groups):
        fold_tr = train_df.iloc[tr_idx]
        stats = fold_tr.groupby(col)[target_col].agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (
            stats["count"] + smoothing
        )
        val_keys = train_df.iloc[val_idx][col]
        oof.iloc[val_idx] = val_keys.map(smoothed).fillna(global_mean).values

    # full-train mapping for use on the actual test set
    full_stats = train_df.groupby(col)[target_col].agg(["mean", "count"])
    full_smoothed = (
        full_stats["mean"] * full_stats["count"] + global_mean * smoothing
    ) / (full_stats["count"] + smoothing)

    return oof, full_smoothed, global_mean

In [205]:
_quintile_median = train["school_quintile"].median()
train = engineer_features(train, quintile_median=_quintile_median )
test = engineer_features(test, quintile_median=_quintile_median)

train = add_seasonality_features(train)
test = add_seasonality_features(test)

train, test = add_frequency_encoding(train, test, "municipality")
train, test = collapse_rare_categories(train, test, "education_level", min_count=100)

# --- combined interaction categoricals -------------------------------
train = make_interaction_categorical(train, "gender", "status_broad_lag",
                                      "gender_x_status_lag")
test = make_interaction_categorical(test, "gender", "status_broad_lag",
                                     "gender_x_status_lag")

train = make_interaction_categorical(train, "education_level", "status_broad_lag",
                                      "education_x_status_lag")
test = make_interaction_categorical(test, "education_level", "status_broad_lag",
                                     "education_x_status_lag")

# race x education_level: sparser combo, so collapse rare cells using
# TRAIN-only counts to avoid leakage.
train = make_interaction_categorical(train, "race", "education_level",
                                      "race_x_education", min_count=50,
                                      train_ref=train)
test = make_interaction_categorical(test, "race", "education_level",
                                     "race_x_education")

# map test's raw combos through the same keep-set as train (anything not
# seen with min_count in train becomes "Other")
_keep_race_edu = set(train["race_x_education"].unique()) - {"Other"}
test["race_x_education"] = test["race_x_education"].where(
    test["race_x_education"].isin(_keep_race_edu), "Other"
)

# --- out-of-fold municipality employment-rate target encoding -------
groups_train = train[ID_COL]
oof_muni_rate, full_muni_rate_map, global_rate = kfold_target_encode(
    train, "municipality", TARGET, groups_train
)
train["municipality_emp_rate"] = oof_muni_rate
test["municipality_emp_rate"] = (
    test["municipality"].map(full_muni_rate_map).fillna(global_rate)
)


In [ ]:
base_cat = [
    "gender",
    "race",
    "province",
    "education_level",
    "race_x_education",
]

ret_cat = base_cat + [
    "status_broad_lag",
    "gender_x_status_lag",
    "education_x_status_lag",
]
# --- 1. Returning Respondents Features ---
ret_numeric = [
    "age",
    "age_sq",
    "employed_lag_x_recency",
    "log_tenure_lag_if_employed",
    "tenure_lag_missing",
    "total_historical_rounds",
    "age_x_employed_lag",
    "municipality_freq",
    "municipality_emp_rate",
    "month_sin",
    "month_cos",
    "month_sin_x_employed_lag",
    "month_cos_x_employed_lag",
]

new_numeric = [
    "age",
    "age_sq",
    "work_readiness_score",
    "municipality_freq",
    "municipality_emp_rate",
    "month_sin",
    "month_cos",
]
new_cat = base_cat

def build_pipeline(numeric_cols, categorical_cols, C=1.0, l1_ratio=1.0):
    num_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
    ])
    prep = ColumnTransformer([
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ])
    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="saga",
        l1_ratio=l1_ratio,
        C=C,
        random_state=RANDOM_STATE,
    )
    return Pipeline([("preprocess", prep), ("clf", clf)])

Cross-Validation

In [207]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(train))
fold_aucs = []
ret_aucs = []
new_aucs = []

for fold, (tr_idx, val_idx) in enumerate(sgkf.split(train, train[TARGET], train[ID_COL]), 1):
    tr_df = train.iloc[tr_idx].copy()
    val_df = train.iloc[val_idx].copy()

    # Recompute fold-level frequency & target encodings
    tr_df, val_df = add_frequency_encoding(tr_df, val_df, "municipality")
    tr_df, val_df = collapse_rare_categories(tr_df, val_df, "education_level", min_count=100)
    
    oof_rate, full_rate_map, g_mean = kfold_target_encode(
        tr_df, "municipality", TARGET, tr_df[ID_COL]
    )
    tr_df["municipality_emp_rate"] = oof_rate
    val_df["municipality_emp_rate"] = val_df["municipality"].map(full_rate_map).fillna(g_mean)

    # Masks for returning vs new entrants
    tr_ret_mask = (tr_df["has_history"] == 1).values
    tr_new_mask = (tr_df["has_history"] == 0).values
    val_ret_mask = (val_df["has_history"] == 1).values
    val_new_mask = (val_df["has_history"] == 0).values

    # Model 1: Returning
    pipe_ret = build_pipeline(ret_numeric, ret_cat, C=0.2, l1_ratio=0.9)
    pipe_ret.fit(tr_df[tr_ret_mask], tr_df.loc[tr_ret_mask, TARGET].astype(int))
    val_probs_ret = pipe_ret.predict_proba(val_df[val_ret_mask])[:, 1]

    # Model 2: New Entrants
    pipe_new = build_pipeline(new_numeric, new_cat, C=0.1, l1_ratio=0.5)
    pipe_new.fit(tr_df[tr_new_mask], tr_df.loc[tr_new_mask, TARGET].astype(int))
    val_probs_new = pipe_new.predict_proba(val_df[val_new_mask])[:, 1]

    # Combine fold predictions
    val_probs = np.zeros(len(val_df))
    val_probs[val_ret_mask] = val_probs_ret
    val_probs[val_new_mask] = val_probs_new
    oof_preds[val_idx] = val_probs

    # Evaluate fold metrics
    y_val = val_df[TARGET].astype(int)
    auc_total = roc_auc_score(y_val, val_probs)
    auc_ret = roc_auc_score(y_val[val_ret_mask], val_probs_ret)
    auc_new = roc_auc_score(y_val[val_new_mask], val_probs_new)
    
    fold_aucs.append(auc_total)
    ret_aucs.append(auc_ret)
    new_aucs.append(auc_new)
    
    print(f"Fold {fold}: Total AUC={auc_total:.5f} | Returning={auc_ret:.5f} | New={auc_new:.5f}")

print(f"\nOverall OOF AUC: {roc_auc_score(train[TARGET].astype(int), oof_preds):.5f}")
print(f"Mean Fold AUC:   {np.mean(fold_aucs):.5f} (+/- {np.std(fold_aucs):.5f})")

Fold 1: Total AUC=0.64513 | Returning=0.71229 | New=0.60998
Fold 2: Total AUC=0.65095 | Returning=0.68537 | New=0.63665
Fold 3: Total AUC=0.65185 | Returning=0.70070 | New=0.62580
Fold 4: Total AUC=0.64705 | Returning=0.68188 | New=0.62972
Fold 5: Total AUC=0.63845 | Returning=0.69800 | New=0.60143

Overall OOF AUC: 0.64634
Mean Fold AUC:   0.64668 (+/- 0.00480)


Fit the pipeline on the full training data and make predictions on the test set:

In [208]:
# Train-wide target encoding for full fit
train_df = train.copy()
test_df = test.copy()

train_df, test_df = add_frequency_encoding(train_df, test_df, "municipality")
train_df, test_df = collapse_rare_categories(train_df, test_df, "education_level", min_count=100)
oof_rate, full_rate_map, g_mean = kfold_target_encode(
    train_df, "municipality", TARGET, train_df[ID_COL]
)
train_df["municipality_emp_rate"] = oof_rate
test_df["municipality_emp_rate"] = test_df["municipality"].map(full_rate_map).fillna(g_mean)

# Full train splits
mask_train_ret = (train_df["has_history"] == 1).values
mask_train_new = (train_df["has_history"] == 0).values

mask_test_ret = (test_df["has_history"] == 1).values
mask_test_new = (test_df["has_history"] == 0).values

# Fit Model 1 (Returning)
pipe_ret = build_pipeline(ret_numeric, ret_cat, C=0.2, l1_ratio=0.9)
pipe_ret.fit(train_df[mask_train_ret], train_df.loc[mask_train_ret, TARGET].astype(int))
test_preds_ret = pipe_ret.predict_proba(test_df[mask_test_ret])[:, 1]

# Fit Model 2 (New Entrants)
pipe_new = build_pipeline(new_numeric, new_cat, C=0.1, l1_ratio=0.5)
pipe_new.fit(train_df[mask_train_new], train_df.loc[mask_train_new, TARGET].astype(int))
test_preds_new = pipe_new.predict_proba(test_df[mask_test_new])[:, 1]

# Assemble final submission
final_test_preds = np.zeros(len(test_df))
final_test_preds[mask_test_ret] = test_preds_ret
final_test_preds[mask_test_new] = test_preds_new

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    "employed_prob": final_test_preds,
})
submission.to_csv("Submissions/Two_Model_Split.csv", index=False)
print("Saved Submissions/Two_Model_Split.csv successfully.")

Saved Submissions/Two_Model_Split.csv successfully.
